# Introduction

This is a preliminary tutorial for the preliminary release of the SBI MRI package. Please do not use it for any official purposes yet, as it still needs to be validated. However, several quality of life improvements are there that you can use to speed up current implementations.

Importantly the package is split into two fundamental parts:
 - the models file
 - the SBI file

The models file is where most of the new stuff is - so I would be quite careful there. The SBI part is relatively standard, including on GPU speed up etc. So if you want to use one part of this package please start with that first.

# Models

In [14]:
from Models import *

%load_ext autoreload

%autoreload 2

from dipy.data import get_fnames
from dipy.core.gradients import gradient_table
from dipy.io.gradients import read_bvals_bvecs
from dipy.core.sphere import disperse_charges, Sphere, HemiSphere


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


This part of the package was built to allow for modular construction of dwMRI models. It means you instantiate a model and then you can add compartments to it as you see fit. Lets start by defining a random gradient table. This is taking directly from dipy and is a overview of your acquisition. If you are working multi diffusion time it is quite important to have that in there as follows:

In [5]:
fimg_init, fbvals, fbvecs = get_fnames(name='small_64D') # Just an example set 
bvals, bvecs = read_bvals_bvecs(fbvals, fbvecs) 
hsph_initial = HemiSphere(xyz=bvecs[1:])
hsph_updated,_ = disperse_charges(hsph_initial,5000)
bvecs = np.vstack([[0,0,0],hsph_updated.vertices])
bvalsExt = np.hstack([bvals, 3000*np.ones_like(bvals)])
bvecsExt = np.vstack([bvecs, bvecs])
bvalsExt[65] = 0

In [6]:
gtabSim = gradient_table(bvals=bvalsExt, bvecs=bvecsExt,small_delta = 0.007*np.ones_like(bvalsExt), big_delta = 0.0173*np.ones_like(bvalsExt))

Note please the: ```small_delta = 0.007*np.ones_like(bvalsExt), big_delta = 0.0173*np.ones_like(bvalsExt)```
Here we have all the same $\delta$ and $\Delta$, but if you have changing deltaas please make sure this is reflected in the vector.

Then we can define a model as follows

In [11]:
DTIModel = Model(gtabSim)
DTIModel._add_compartment("DTI","Hindered") # The first "DTI" is what kind of model we want and "Hindered" is the name we give it
DTIModel._set_default_SNR(20)

# or

AxCaliberModel = Model(gtabSim)
AxCaliberModel._add_compartment("DTI","Hindered")
AxCaliberModel._add_compartment("Sticks","Axons",Size='Infer')
AxCaliberModel._add_compartment("Sticks","Dendrites",Dispersion=True,Size='Infer')
AxCaliberModel._set_default_SNR(50)

# or

StickAndBallModel = Model(gtabSim)
StickAndBallModel._add_compartment("DTI","Hindered")
StickAndBallModel._add_compartment("Sticks","Axons",Size='Infer')
StickAndBallModel._add_compartment("Balls","Astrocytes")
StickAndBallModel._set_default_SNR(50)

# or 

StickAndBallFWModel = Model(gtabSim)
StickAndBallFWModel._add_compartment("DTI","Hindered")
StickAndBallFWModel._add_compartment("Sticks","Axons",Size='Infer')
StickAndBallFWModel._add_compartment("Balls","Astrocytes")
StickAndBallFWModel._add_compartment("FreeWater","Free Water")
StickAndBallFWModel._set_default_SNR(50)

*****WARNING: You are trying to fit size with only 1 diffusion time - proceed with caution!*****
*****WARNING: You are trying to fit size with only 1 diffusion time - proceed with caution!*****
*****WARNING: You are trying to fit size with only 1 diffusion time - proceed with caution!*****
*****WARNING: You are trying to fit size with only 1 diffusion time - proceed with caution!*****
*****WARNING: You are trying to fit size with only 1 diffusion time - proceed with caution!*****
*****WARNING: You are trying to fit size with only 1 diffusion time - proceed with caution!*****


All of the compartments have slightly arguments - for now helper functions are not defined, so if you want to know the arguments you can look into the code. The helper functions will come when im back. 

If you then want simulate these models its as easy as:

In [17]:
Params,Signals = StickAndBallFWModel.simulation(10_000,parallel=True)

Simulating compartments: 4it [00:34,  8.51s/it]

File was saved as DSBFW_20260619_1440.h5!


Conveniently, the models build your parameter list dynamically and can be accessed as:

In [18]:
StickAndBallFWModel._get_parameter_names

['Hindered_f',
 'Axons_f',
 'Astrocytes_f',
 'Free Water_f',
 'Hindered_D_xx',
 'Hindered_D_xy',
 'Hindered_D_yy',
 'Hindered_D_xz',
 'Hindered_D_yz',
 'Hindered_D_zz',
 'Axons_theta',
 'Axons_phi',
 'Axons_D_par',
 'Axons_D_perp',
 'Axons_size',
 'Astrocytes_D_sph',
 'Astrocytes_size']

For memory purposes its best to always use parallel = True - but currently the n_jobs is set to -1. So if you want to run on the cluster and dont mind waiting a bit you can set parallel to False.

Now you might have noticed that a file was saved - this is saved as DSBFW_todaysdat_timenow.h5. The DSBFW describes the model you are running and changes based on the compartments you have: 
- D = DTI
- S = Sticks
- B = Balls
- FW = Freewater

This way you can generate simulations and they will be automatically saved for reuse. If you want to add a random seed, its as simple as  

In [19]:
Params,Signals = StickAndBallFWModel.simulation(10_000,parallel=True,rng = np.random.default_rng(42))

Simulating compartments: 4it [00:33,  8.34s/it]

File was saved as DSBFW_20260619_1445.h5!


By accident, we now overwrote the previous simulations - if we want to get them back there is a useful little function:

In [20]:
P,S,S_raw,names,snr = Helpers.read_h5('DSBFW_20260619_1440.h5') #(put the name of the file from before

Also right now the snr is set to a default 50 for the model (._set_default_SNR(50)). 
if you want to change it just put in: 

In [21]:
Params,Signals = StickAndBallFWModel.simulation(10_000,parallel=True,rng = np.random.default_rng(42),custom_snr=30)

Simulating compartments: 4it [00:33,  8.28s/it]

File was saved as DSBFW_20260619_1448.h5!


# SBI

The SBI part will be something all of you are more familiar with. All i have done is put things in one place and sped some other things up. So how does this work in our current set-up.

We have generated some parameters and associated signals S. First we need to make sure that our data is in the right format, and that we don't include redundant information

In [44]:
import Helpers #Custom functions
import SBI

In [45]:
Par,Obs,Names,Bounds = Helpers.PrepData(P,S,names)

As one of the fractions is redundant we will remove it (1-sum of the rest). We are removing: Hindered_f


Next we train the network

In [46]:
Network = SBI.Train_Network_gpu(Par,Obs) #this will automatically see if you have speed-up possibilities (either GPU or mac MPS)

/Users/maximilianeggl/miniconda3/envs/SBI_MRI/lib/python3.14/site-packages/sbi/inference/trainers/npe/npe_base.py:184: UserWarning: Data x has device 'cpu'. Moving x to the data_device 'mps'. Training will proceed on device 'mps'.
  theta, x = validate_theta_and_x(
/Users/maximilianeggl/miniconda3/envs/SBI_MRI/lib/python3.14/site-packages/sbi/inference/trainers/npe/npe_base.py:184: UserWarning: Parameters theta has device 'cpu'. Moving theta to the data_device 'mps'. Training will proceed on device 'mps'.
  theta, x = validate_theta_and_x(


 Neural network successfully converged after 160 epochs.

Lets say now we want to do some inference on an in-silico dataset

In [47]:
Params_test,Signals_test = StickAndBallFWModel.simulation(500,parallel=True,rng = np.random.default_rng(2026),custom_snr=30)

Simulating compartments: 4it [00:08,  2.03s/it]

File was saved as DSBFW_20260619_1458.h5!


In [50]:
Par_t,Obs_t,_,_ = Helpers.PrepData(Params_test,Signals_test,StickAndBallFWModel._get_parameter_names)

As one of the fractions is redundant we will remove it (1-sum of the rest). We are removing: Hindered_f


In [52]:
Result = SBI.Infer(Network,Obs_t)

100%|██████████████████████████████████████████| 16/16 [00:00<00:00, 19.69it/s]


We can also run this over a whole Volume using ```InferFromVolume()```. Here the input is the network, the data (can be 2D or 3D) and an associated Mask. Depending on the size of the data you are passing this might take a bit.